# Day 15: Walk-Forward Modeling

This notebook trains Random Forest and Gradient Boosting regressors using walk-forward validation. Missing feature values are median-imputed inside each fold using the training data only.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

cwd = Path.cwd()
root = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(root / 'src'))

from feature_engineering import compute_all_features, event_based_generator, leakage_audit

data_path = root / 'data' / 'processed' / 'Davao_Earthquakes_with_Dist.csv'
output_dir = root / 'outputs'
output_dir.mkdir(exist_ok=True)

data_path

PosixPath('/home/jshu/SeismoGuard/data/processed/Davao_Earthquakes_with_Dist.csv')

## Load Events

In [2]:
events = pd.read_csv(data_path)
events['province'] = (
    events['location']
    .astype(str)
    .str.extract(r'\(([^)]+)\)')[0]
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
events = events.dropna(subset=['province'])

province_map = {
    'Davao del Norte': 'Davao Del Norte',
    'Davao del Sur': 'Davao Del Sur',
    'Island Garden City of Samal': 'Island Garden City Of Samal',
    'Municipality of Sarangani': 'Municipality Of Sarangani',
}
valid_provinces = [
    'Davao De Oro',
    'Davao Del Norte',
    'Davao Del Sur',
    'Davao Occidental',
    'Davao Oriental',
]

events['province'] = events['province'].str.strip().replace(province_map)
events = events[events['province'].isin(valid_provinces)].copy()

events.shape, events['province'].value_counts()

((19258, 8),
 province
 Davao Oriental      9922
 Davao Occidental    5970
 Davao Del Sur       1704
 Davao De Oro        1266
 Davao Del Norte      396
 Name: count, dtype: int64)

## Build Or Load Feature Matrix

In [3]:
x_path = output_dir / 'feature_matrix_X.pkl'
y_path = output_dir / 'target_y.pkl'

if x_path.exists() and y_path.exists():
    X = pd.read_pickle(x_path)
    y = pd.read_pickle(y_path)
else:
    X, y = event_based_generator(
        events_df=events,
        feature_fn=compute_all_features,
        province_col='province',
        datetime_col='datetime',
        magnitude_col='magnitude',
    )
    X.to_pickle(x_path)
    y.to_pickle(y_path)
    X.to_csv(output_dir / 'feature_matrix_X.csv')
    y.to_csv(output_dir / 'target_y.csv')

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Columns:', list(X.columns))
X.isna().sum().sort_values(ascending=False)

X shape: (1980, 17)
y shape: (1980,)
Columns: ['count_1d', 'count_7d', 'count_30d', 'max_mag_7d', 'max_mag_30d', 'mean_depth_30d', 'days_since_m5', 'min_fault_dist_km', 'mean_fault_dist_km_7d', 'pct_near_fault_30d', 'num_clusters_30d', 'largest_cluster_size_30d', 'pct_clustered_30d', 'max_grid_cell_count_30d', 'b_value_180d', 'a_value_180d', 'delta_b_90d']


delta_b_90d                 957
a_value_180d                481
b_value_180d                481
pct_near_fault_30d          417
max_mag_7d                  417
min_fault_dist_km           417
mean_fault_dist_km_7d       417
mean_depth_30d               67
max_mag_30d                  67
count_1d                      0
days_since_m5                 0
count_30d                     0
count_7d                      0
pct_clustered_30d             0
largest_cluster_size_30d      0
num_clusters_30d              0
max_grid_cell_count_30d       0
dtype: int64

## Leakage Audit

In [4]:
audit = leakage_audit(
    X=X,
    events_df=events,
    feature_fn=compute_all_features,
    province_col='province',
)
mismatches = audit[(~audit['match_full']) | (~audit['match_past'])]
print('Mismatch count:', len(mismatches))
mismatches.head()

Mismatch count: 0


,province,forecast_date,feature,observed,recomputed_full,recomputed_past,match_full,match_past


## Walk-Forward Splits

In [5]:
def make_walk_forward_splits(X, min_train_weeks=104, test_weeks=26):
    dates = pd.Index(X.index.get_level_values('forecast_date').unique()).sort_values()
    splits = []
    start = min_train_weeks

    while start < len(dates):
        train_dates = dates[:start]
        test_dates = dates[start:start + test_weeks]
        if len(test_dates) == 0:
            break

        train_mask = X.index.get_level_values('forecast_date').isin(train_dates)
        test_mask = X.index.get_level_values('forecast_date').isin(test_dates)
        splits.append((train_mask, test_mask, train_dates[-1], test_dates[0], test_dates[-1]))
        start += test_weeks

    return splits

splits = make_walk_forward_splits(X)
len(splits), [(s[2], s[3], s[4]) for s in splits]

(13,
 [(Timestamp('2019-12-23 00:00:00+0000', tz='UTC'),
   Timestamp('2019-12-30 00:00:00+0000', tz='UTC'),
   Timestamp('2020-06-22 00:00:00+0000', tz='UTC')),
  (Timestamp('2020-06-22 00:00:00+0000', tz='UTC'),
   Timestamp('2020-06-29 00:00:00+0000', tz='UTC'),
   Timestamp('2020-12-21 00:00:00+0000', tz='UTC')),
  (Timestamp('2020-12-21 00:00:00+0000', tz='UTC'),
   Timestamp('2020-12-28 00:00:00+0000', tz='UTC'),
   Timestamp('2021-06-21 00:00:00+0000', tz='UTC')),
  (Timestamp('2021-06-21 00:00:00+0000', tz='UTC'),
   Timestamp('2021-06-28 00:00:00+0000', tz='UTC'),
   Timestamp('2021-12-20 00:00:00+0000', tz='UTC')),
  (Timestamp('2021-12-20 00:00:00+0000', tz='UTC'),
   Timestamp('2021-12-27 00:00:00+0000', tz='UTC'),
   Timestamp('2022-06-20 00:00:00+0000', tz='UTC')),
  (Timestamp('2022-06-20 00:00:00+0000', tz='UTC'),
   Timestamp('2022-06-27 00:00:00+0000', tz='UTC'),
   Timestamp('2022-12-19 00:00:00+0000', tz='UTC')),
  (Timestamp('2022-12-19 00:00:00+0000', tz='UTC'),
 

## Train And Evaluate

In [6]:
models = {
    'random_forest': RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    ),
    'gradient_boosting': GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.04,
        max_depth=3,
        min_samples_leaf=5,
        random_state=42,
    ),
}

metrics = []
prediction_frames = []

for fold, (train_mask, test_mask, train_end, test_start, test_end) in enumerate(splits, start=1):
    X_train, X_test = X.loc[train_mask], X.loc[test_mask]
    y_train, y_test = y.loc[train_mask], y.loc[test_mask]

    for model_name, model in models.items():
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', model),
        ])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        metrics.append({
            'fold': fold,
            'model': model_name,
            'train_end': train_end,
            'test_start': test_start,
            'test_end': test_end,
            'train_rows': len(X_train),
            'test_rows': len(X_test),
            'mae': mean_absolute_error(y_test, preds),
            'rmse': np.sqrt(mean_squared_error(y_test, preds)),
            'r2': r2_score(y_test, preds),
        })

        fold_preds = X_test.reset_index()[['province', 'forecast_date']].copy()
        fold_preds['fold'] = fold
        fold_preds['model'] = model_name
        fold_preds['actual'] = y_test.to_numpy()
        fold_preds['prediction'] = preds
        prediction_frames.append(fold_preds)

metrics_df = pd.DataFrame(metrics)
predictions_df = pd.concat(prediction_frames, ignore_index=True)

metrics_df.to_csv(output_dir / 'walk_forward_metrics.csv', index=False)
predictions_df.to_csv(output_dir / 'walk_forward_predictions.csv', index=False)

metrics_df

,fold,model,train_end,test_start,test_end,train_rows,test_rows,mae,rmse,r2
0,1,random_forest,2019-12-23 00:00:00+00:00,2019-12-30 00:00:00+00:00,2020-06-22 00:00:00+00:00,403,126,0.795489,0.997169,0.713455
1,1,gradient_boosting,2019-12-23 00:00:00+00:00,2019-12-30 00:00:00+00:00,2020-06-22 00:00:00+00:00,403,126,0.926724,1.143143,0.623420
2,2,random_forest,2020-06-22 00:00:00+00:00,2020-06-29 00:00:00+00:00,2020-12-21 00:00:00+00:00,529,130,1.020106,1.319492,0.430431
3,2,gradient_boosting,2020-06-22 00:00:00+00:00,2020-06-29 00:00:00+00:00,2020-12-21 00:00:00+00:00,529,130,1.007421,1.270505,0.471938
4,3,random_forest,2020-12-21 00:00:00+00:00,2020-12-28 00:00:00+00:00,2021-06-21 00:00:00+00:00,659,130,1.026196,1.278596,0.469382
5,3,gradient_boosting,2020-12-21 00:00:00+00:00,2020-12-28 00:00:00+00:00,2021-06-21 00:00:00+00:00,659,130,1.012133,1.327135,0.428329
6,4,random_forest,2021-06-21 00:00:00+00:00,2021-06-28 00:00:00+00:00,2021-12-20 00:00:00+00:00,789,130,0.890655,1.172596,0.468677
7,4,gradient_boosting,2021-06-21 00:00:00+00:00,2021-06-28 00:00:00+00:00,2021-12-20 00:00:00+00:00,789,130,0.958669,1.230818,0.414605
8,5,random_forest,2021-12-20 00:00:00+00:00,2021-12-27 00:00:00+00:00,2022-06-20 00:00:00+00:00,919,130,1.045827,1.285520,0.508092
9,5,gradient_boosting,2021-12-20 00:00:00+00:00,2021-12-27 00:00:00+00:00,2022-06-20 00:00:00+00:00,919,130,1.045375,1.314525,0.485644


## Summary

In [7]:
summary = (
    metrics_df
    .groupby('model')[['mae', 'rmse', 'r2']]
    .agg(['mean', 'std'])
    .round(4)
)
summary.to_csv(output_dir / 'walk_forward_summary.csv')
summary

mae            rmse              r2        
                     mean     std    mean     std    mean     std
model                                                            
gradient_boosting  0.9581  0.0961  1.2392  0.1246  0.4032  0.1290
random_forest      0.9350  0.0990  1.2068  0.1240  0.4320  0.1238